In [0]:
import requests

url = "https://download.geonames.org/export/dump/cities500.zip"

response = requests.get(url)
response.raise_for_status()

with open("data/cities500.zip", "wb") as f:
    f.write(response.content)

In [0]:
import zipfile

with zipfile.ZipFile("data/cities500.zip", "r") as zip_ref:
    zip_ref.extractall("data/geonames")

## Read it with PySpark

In [0]:
columns = [
    "geoname_id",
    "city_name",
    "ascii_name",
    "alternate_names",
    "latitude",
    "longitude",
    "feature_class",
    "feature_code",
    "country_code",
    "cc2",
    "admin1_code",
    "admin2_code",
    "admin3_code",
    "admin4_code",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification_date"
]

df = (
    spark.read
    .option("sep", "\t")
    .option("header", "true")
    .option("inferSchema", "true")
    #.csv("data/geonames/cities500.txt")
    .csv("data/cities.txt")
)

df = df.toDF(*columns)


In [0]:
display(df)

In [0]:
country_url = "https://download.geonames.org/export/dump/countryInfo.txt"

response = requests.get(country_url)
response.raise_for_status()

with open("data/countryInfo.txt", "wb") as f:
    f.write(response.content)

In [0]:
country_df = (
    spark.read
    .option("sep", "\t")
    .option("comment", "#")
    .option("header", "false")
    .csv("/data/countryInfo.txt")
)

In [0]:
cities = df.select(
    "geoname_id",
    "city_name",
    "country_code",
    "latitude",
    "longitude",
    "population",
    "timezone"
)

In [0]:
cities_with_country = cities.join(
    country_df,
    cities.country_code == country_df.country_code,
    "left"
)

In [0]:
(
    cities_with_country
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("reference.geonames_cities")
)